### Libraries

In [1]:
# Models deep
import torch
import torch.nn as nn

# Data
from torch.utils.data import DataLoader

# Numerical
import numpy as np


# Local libraries
from utils.torch_lib.TabularTransformer import TabularTransformer, TransactionDataset



Checking if a faster device is available

In [2]:

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [3]:
learning_rate = 1e-3
batch_size = 128
epochs = 2

## Importing data

In [4]:

CSV_TRAIN_PATH = '../../data_ieee/transactions_train.csv'
CSV_TEST_PATH = '../../data_ieee/transactions_test.csv'
PKL_PATH = '../../models/preprocessing_config.pkl'


train_dataset = TransactionDataset(CSV_TRAIN_PATH, PKL_PATH, target_col='isFraud')
test_dataset = TransactionDataset(CSV_TEST_PATH, PKL_PATH, target_col='isFraud')

loader_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
loader_test = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

for x_cat, x_cont, labels in loader_train:
    print(f"Categorical Batch Shape: {x_cat.shape}") 
    print(f"Continuous Batch Shape: {x_cont.shape}")  
    print(f"Labels Shape: {labels.shape}")            
    break

Preprocessing data... this may take a moment.
Preprocessing data... this may take a moment.
Categorical Batch Shape: torch.Size([128, 33])
Continuous Batch Shape: torch.Size([128, 170])
Labels Shape: torch.Size([128])


## Defining testing and training

In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    model.train()
    for batch, (X_cat, X_cont, y) in enumerate(dataloader):

        X_cat, X_cont, y = X_cat.to(device), X_cont.to(device), y.to(device)

        pred = model(X_cat, X_cont)
        loss = loss_fn(pred, y)


        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X_cat)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):


    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0


    with torch.no_grad():
        for (X_cat, X_cont, y) in dataloader:

            X_cat, X_cont, y = X_cat.to(device), X_cont.to(device), y.to(device)

            pred = model(X_cat, X_cont)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## Defining the model

In [6]:
# Geting the dimentions
n_categories = train_dataset.get_n_categories()
n_continuous = train_dataset.num_idx[1]

model = TabularTransformer(n_categories=n_categories, n_continuous = n_continuous, n_classes = 2, embed_dim = 16)
model.to(device)

TabularTransformer(
  (embeddings): ModuleList(
    (0): Embedding(461398, 16)
    (1): Embedding(6, 16)
    (2): Embedding(12822, 16)
    (3-4): 2 x Embedding(6, 16)
    (5): Embedding(61, 16)
    (6): Embedding(62, 16)
    (7-9): 3 x Embedding(4, 16)
    (10): Embedding(5, 16)
    (11-16): 6 x Embedding(4, 16)
    (17): Embedding(5, 16)
    (18): Embedding(4, 16)
    (19): Embedding(5, 16)
    (20-22): 3 x Embedding(4, 16)
    (23): Embedding(77, 16)
    (24): Embedding(126, 16)
    (25): Embedding(238, 16)
    (26): Embedding(6, 16)
    (27-31): 5 x Embedding(4, 16)
    (32): Embedding(1688, 16)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
        )
        (linear1): Linear(in_features=16, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear

## Defining the loss and optimization algorithm

In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Training loop

In [8]:

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(loader_train, model, loss_fn, optimizer)
    test_loop(loader_test, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.709700  [  128/472432]
loss: 0.165955  [12928/472432]
loss: 0.142690  [25728/472432]
loss: 0.192427  [38528/472432]
loss: 0.152896  [51328/472432]
loss: 0.153470  [64128/472432]
loss: 0.032818  [76928/472432]
loss: 0.075743  [89728/472432]
loss: 0.197700  [102528/472432]
loss: 0.141966  [115328/472432]
loss: 0.168528  [128128/472432]
loss: 0.072597  [140928/472432]
loss: 0.195182  [153728/472432]
loss: 0.198792  [166528/472432]
loss: 0.092686  [179328/472432]
loss: 0.153872  [192128/472432]
loss: 0.094238  [204928/472432]
loss: 0.165301  [217728/472432]
loss: 0.119561  [230528/472432]
loss: 0.055363  [243328/472432]
loss: 0.097494  [256128/472432]
loss: 0.118705  [268928/472432]
loss: 0.213426  [281728/472432]
loss: 0.119893  [294528/472432]
loss: 0.100737  [307328/472432]
loss: 0.119936  [320128/472432]
loss: 0.094194  [332928/472432]
loss: 0.105761  [345728/472432]
loss: 0.093760  [358528/472432]
loss: 0.233542  [371328/472432]
loss: 0.

## Saving and checking 

### Model result (with no importing)

In [13]:

x_cat = test_dataset.Xp_cat[0:1, :].to(device)
x_cont = test_dataset.Xp_cont[0:1, :].to(device)
y = test_dataset.y[0:1].to(device)

model.eval()
with torch.no_grad():
    pred = model(x_cat, x_cont)

print(f"Predicción para el primer elemento: {pred} | Actual : {y}")

Predicción para el primer elemento: tensor([[ 1.9898, -2.2738]], device='cuda:0') | Actual : tensor([0], device='cuda:0')


### Saving the model

In [10]:
torch.save(model.state_dict(), '../../models/TabularTransformer_weights.pth')

### Importing the model

In [11]:
model = TabularTransformer(n_categories=n_categories, n_continuous = n_continuous, n_classes = 2, embed_dim = 16)
model.load_state_dict(torch.load('../../models/TabularTransformer_weights.pth', weights_only=True))

<All keys matched successfully>

### Model result (imported)

In [12]:
x_cat = test_dataset.Xp_cat[0:1, :].to(device)
x_cont = test_dataset.Xp_cont[0:1, :].to(device)
y = test_dataset.y[0:1].to(device)
model.to(device)
model.eval()
with torch.no_grad():
    pred = model(x_cat, x_cont)

print(f"Predicción para el primer elemento: {pred} | Actual : {y}")

Predicción para el primer elemento: tensor([[ 1.9898, -2.2738]], device='cuda:0') | Actual : tensor([0], device='cuda:0')
